In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
def clean_and_preprocess_data(df: pd.DataFrame, fill_strategy: str = 'median') -> pd.DataFrame:
    cleaned_df = df.copy()
    
    if fill_strategy == 'median':
        cleaned_df["ca"] = cleaned_df["ca"].fillna(cleaned_df["ca"].median())
        cleaned_df["thal"] = cleaned_df["thal"].fillna(cleaned_df["thal"].median())
    else:
        cleaned_df["ca"] = cleaned_df["ca"].fillna(0)
        cleaned_df["thal"] = cleaned_df["thal"].fillna(0)

    cleaned_df["target"] = cleaned_df["target"].apply(lambda x: 1 if x > 0 else 0)

    Q1 = cleaned_df['chol'].quantile(0.25)
    Q3 = cleaned_df['chol'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    cleaned_df['chol'] = np.clip(cleaned_df['chol'], lower_bound, upper_bound)

    cleaned_df = pd.get_dummies(cleaned_df, columns=['cp', 'restecg', 'slope'], drop_first=True)

    scaler = StandardScaler()
    num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
    cleaned_df[num_cols] = scaler.fit_transform(cleaned_df[num_cols])

    return cleaned_df

In [ ]:
column_names = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", 
                "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]

raw_df = pd.read_csv("https://uci.edu", 
                     names=column_names, na_values="?")

final_df = clean_and_preprocess_data(raw_df)
final_df.to_csv("cleaned_heart.csv", index=False)
print("Notebook 1 Complete: 'cleaned_heart.csv' created.")